## Синтез речи при помощи обученной модели Tacotron2

В данном блокноте модель, обученная в блокноте Lab3_train, используется для синтеза речи на основе переданных в неё предложений. Некоторые комментарии к коду см. в блокноте Lab3_train.

In [183]:
import os
import re
import gc
from tqdm import tqdm
import pickle
import ast
from dataclasses import dataclass, field

import pandas as pd
import torchaudio
import IPython
import torch
from IPython.display import Audio, display
import matplotlib.pyplot as plt

import pysbd

In [2]:
tqdm.pandas()

In [3]:
from TTS.api import TTS
from TTS.tts.configs.shared_configs import BaseDatasetConfig, BaseAudioConfig

C:\python_venv\venv1_py12\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from trainer import Trainer, TrainerArgs
from TTS.tts.configs.tacotron2_config import Tacotron2Config
from TTS.tts.configs.shared_configs import CharactersConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.tacotron2 import Tacotron2
from TTS.tts.utils.text.tokenizer import TTSTokenizer
from TTS.utils.audio import AudioProcessor
from TTS.utils.synthesizer import Synthesizer

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
class Characters():
    def __init__(self, num_chars):
        self.num_chars = num_chars

In [7]:
class TokensEncoder():
    def __init__(self):
        self.tokens_count = 4
        self.tokens_to_ids = {"<PAD>": 0, "<BOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.pad_id = 0
        self.use_phonemes = False
        
    
    def fit(self, df_column):
        for index, row in df_column.items():
            for token in row:
                if token not in self.tokens_to_ids:
                    self.tokens_to_ids[token] = self.tokens_count
                    self.tokens_count += 1

        self.ids_to_tokens = {value: key for key, value in self.tokens_to_ids.items()}
        self.characters = Characters(self.tokens_count)

    def transform(self, df_column):
        return df_column.apply(lambda x: [self.tokens_to_ids[token] if token in self.tokens_to_ids else self.tokens_to_ids["<UNK>"] for token in x])

    def transform_one(self, tokens):
        return [self.tokens_to_ids[token] if token in self.tokens_to_ids else self.tokens_to_ids["<UNK>"] for token in tokens]

    def inverse_transform(self, df_column):
        return df_column.apply(lambda x: [self.ids_to_tokens[id] for id in x])

    def inverse_transform_one(self, ids):
        return [self.ids_to_tokens[id] for id in ids]

    def text_to_ids(self, text, language=None):
        while isinstance(text, str):
            try:
                text = pd.eval(text)
            except:
                print(text)
                raise
                

        return [self.tokens_to_ids[token] for token in text]

    def print_logs(self, log_level):
        pass

In [8]:
def add_bos_eos(df_part, column_name):
    df_part[column_name] = df_part[column_name].apply(lambda x: ["<BOS>"] + x + ["<EOS>"])

In [9]:
def add_bos_eos_to_list(word):
    return ["<BOS>"] + word + ["<EOS>"]

In [10]:
def remove_bos_eos_from_list(word):
    while "<BOS>" in word:
        word.remove("<BOS>")
    
    while "<EOS>" in word:
        word.remove("<EOS>")
    
    return word

In [11]:
def remove_bos_eos(df_part, column_name):
    df_part[column_name] = df_part[column_name].apply(remove_bos_eos_from_list)

In [12]:
def remove_pad_from_list(word):
    while "<PAD>" in word:
        word.remove("<PAD>")
    
    return word

In [13]:
def remove_pad(df_part, column_name):
    df_part[column_name] = df_part[column_name].apply(remove_pad_from_list)

In [14]:
def train_and_apply_tokens_encoder(tokens_encoder, train_df, val_df, column):
    tokens_encoder.fit(train_df[column])
    train_df[column] = tokens_encoder.transform(train_df[column])
    val_df[column] = tokens_encoder.transform(val_df[column])

In [15]:
class WordsDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.df = df
        self.words = [torch.LongTensor(x).to(device) for x in tqdm(df["word"].values)]
        self.allophones = [torch.LongTensor(x).to(device) for x in tqdm(df["allophones"].values)]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        return self.words[index], self.allophones[index]


In [16]:
class WordsPadder():
    def __init__(self, pad_token):
        self.pad_token = pad_token

    def __call__(self, batch):
        words, allophone_sequences = zip(*batch)
        
        padded_words = torch.nn.utils.rnn.pad_sequence(words, padding_value=self.pad_token)
        words_pad_mask = (padded_words == self.pad_token)
        padded_allophone_sequences = torch.nn.utils.rnn.pad_sequence(allophone_sequences, padding_value=self.pad_token)
        allophones_pad_mask = (padded_allophone_sequences == self.pad_token)
        
        return padded_words, padded_allophone_sequences, words_pad_mask.transpose(-1, -2), allophones_pad_mask.transpose(-1, -2)



In [17]:
class Seq2seq(torch.nn.Module):
    def __init__(self, word_tokens_count, allophone_tokens_count, transformer_dim=512, max_seq_len=100, dropout_prob=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.words_emb = torch.nn.Embedding(word_tokens_count, transformer_dim)
        self.words_positional_encoding = torch.nn.Embedding(self.max_seq_len, transformer_dim)
        self.allophones_emb = torch.nn.Embedding(allophone_tokens_count, transformer_dim)
        self.allophones_positional_encoding = torch.nn.Embedding(self.max_seq_len, transformer_dim)
        self.dropout = torch.nn.Dropout(dropout_prob)
        self.transformer = torch.nn.Transformer(d_model=transformer_dim, dropout=dropout_prob)
        self.linear = torch.nn.Linear(transformer_dim, allophone_tokens_count)
    
    def forward(self, words, allophones, words_pad_mask, allophones_pad_mask):
        words = self.words_emb(words)
        words += self.words_positional_encoding(torch.arange(words.shape[0]).to(device)).unsqueeze(1)
        words = self.dropout(words)
        
        allophones = self.allophones_emb(allophones)
        allophones += self.allophones_positional_encoding(torch.arange(allophones.shape[0]).to(device)).unsqueeze(1)
        allophones = self.dropout(allophones)
        result = self.transformer(words, allophones, tgt_mask=torch.nn.Transformer.generate_square_subsequent_mask(allophones.shape[0]).to(device), src_key_padding_mask=words_pad_mask, tgt_key_padding_mask=allophones_pad_mask)
        return self.linear(result)
    
    def words_to_allophones(self, word, max_length=None):
        if max_length is None:
            max_length = self.max_seq_len
        
        with torch.no_grad():
            allophones = torch.ones((1, word.size(1)), device=device).long()
            finished = torch.zeros(word.size(1), device=device).bool()

            for i in range(0, max_length):
                if finished.all():
                    break
                
                output = self.forward(word, allophones, None, (allophones == 0).transpose(-1, -2))
                
                next_tokens = output.argmax(dim=-1)[output.shape[0] - 1:]
                
                allophones = torch.cat([allophones, next_tokens], dim=0)
                
                finished = finished | (next_tokens.squeeze(0) == 2)
            
            return allophones
    
    def word_to_allophones(self, word):
        allophones = torch.ones((1, 1)).to(device).long()
        while allophones[-1, 0] != 2 and len(allophones) < self.max_seq_len:
            result = self.forward(word, allophones, None, (allophones == 0).transpose(-1, -2))
            allophones = torch.cat([allophones, result.argmax(dim=-1)[result.shape[0] - 1, :].unsqueeze(1)], dim=0)
    
        return allophones


In [18]:
def split_letters_and_symbols(word):
    symbols = ""
    for i in range(len(word) - 1, -1, -1):
        if word[i].isalpha():
            return word[0:i + 1], symbols

        symbols = word[i] + symbols

    return "", symbols

In [19]:
def batch_predict(model, dataloader):
    model.eval()
    all_predictions = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader):
            words, _, words_pad_mask, _ = batch
            predictions = model.words_to_allophones(words, 50)
            
            # Конвертируем в numpy и сохраняем
            for i in range(words.size(1)):
                word_seq = words[:, i].cpu().numpy()
                pred_seq = predictions[:, i].cpu().numpy()
                all_predictions.append((word_seq, pred_seq))

            gc.collect()
            torch.cuda.empty_cache()
    
    return all_predictions

In [20]:
def get_pause(punctuation):
    if punctuation == ',':
        return [' ', '-']
    
    if punctuation == '.' or punctuation == '?' or punctuation == '!':
        return [' ', '-', '-']

    return [' ']

In [21]:
def transform_word(word, word_tokens_encoder, model, allophones_tokens_encoder, new_allophones_encoder):
    word = word.lower()
    tokenized_word = word_tokens_encoder.transform_one(add_bos_eos_to_list(list(word)))
    tokenized_allophones = model.word_to_allophones(torch.LongTensor(tokenized_word).to(device).unsqueeze(1))
    allophones = remove_bos_eos_from_list(allophones_tokens_encoder.inverse_transform_one(tokenized_allophones.flatten().cpu().tolist()))
    return new_allophones_encoder.transform_one(allophones)

In [22]:
def transform_sentence(sentence, word_tokens_encoder, model, allophones_tokens_encoder, new_allophones_encoder):
    sentence = re.sub("[^а-яА-Я,.!?]", " ", sentence)
    words = re.sub(" +", " " , sentence.lower()).split()
    punctuations = [split_letters_and_symbols(word)[1] for word in words]
    words = [split_letters_and_symbols(word)[0] for word in words]
    
    allophones = [transform_word(word, word_tokens_encoder, model, allophones_tokens_encoder, new_allophones_encoder) for word in words]
    pauses = [new_allophones_encoder.transform_one(get_pause(punctuation)) for punctuation in punctuations]
    punctuations = [new_allophones_encoder.transform_one(punctuation) for punctuation in punctuations]

    sentence = []
    for i in range(len(allophones)):
        sentence += allophones[i]
        sentence += punctuations[i]
        sentence += pauses[i]

    return sentence

In [23]:
class SpecialTokenizer():
    def __init__(self, word_tokens_encoder, model, allophones_tokens_encoder, new_allophones_encoder):
        self.word_tokens_encoder = word_tokens_encoder
        self.model = model
        self.allophones_tokens_encoder = allophones_tokens_encoder
        self.new_allophones_encoder = new_allophones_encoder
        
        self.tokens_count = 4
        self.tokens_to_ids = {0: 0, 1: 1, 2: 2, 3: 3}
        self.pad_id = 0
        self.use_phonemes = False
        
    
    def fit(self, df_column):
        for index, row in df_column.items():
            for token in row:
                if token not in self.tokens_to_ids:
                    self.tokens_to_ids[token] = self.tokens_count
                    self.tokens_count += 1

        self.ids_to_tokens = {value: key for key, value in self.tokens_to_ids.items()}
        self.characters = Characters(self.tokens_count)

    def transform(self, df_column):
        return df_column.apply(lambda x: [self.tokens_to_ids[token] if token in self.tokens_to_ids else self.tokens_to_ids["<UNK>"] for token in x])

    def inverse_transform(self, df_column):
        return df_column.apply(lambda x: [self.ids_to_tokens[id] for id in x])

    def text_to_ids(self, text, language=None):
        if type(text) == list:
            return text
        
        if text[0] == '[':
            return ast.literal_eval(text)

        return transform_sentence(text, self.word_tokens_encoder, self.model, self.allophones_tokens_encoder, self.new_allophones_encoder)
        
    def print_logs(self, log_level):
        pass

In [24]:
loaded_model = torch.load("model.pt", weights_only=False).to(device)
loaded_model.eval()

with open("word_tokens_encoder.pickle", "rb") as file:
    loaded_word_tokens_encoder = pickle.load(file)

with open("allophones_tokens_encoder.pickle", "rb") as file:
    loaded_allophones_tokens_encoder = pickle.load(file)

with open("new_allophones_encoder.pickle", "rb") as file:
    loaded_new_allophones_encoder = pickle.load(file)

In [25]:
loaded_pred_df = pd.read_csv("data/tokenized_allophones_metadata_RUSLAN_22200.csv", delimiter='|', header=None)

In [26]:
tokenizer = SpecialTokenizer(loaded_word_tokens_encoder, loaded_model, loaded_allophones_tokens_encoder, loaded_new_allophones_encoder)
tokenizer.fit(loaded_pred_df[1].progress_apply(pd.eval))

100%|███████████████████████████████████████████████████████████████████████████| 21374/21374 [00:43<00:00, 492.00it/s]


In [27]:
 dataset_config = BaseDatasetConfig(
        formatter="ruslan", meta_file_train="tokenized_allophones_metadata_RUSLAN_22200.csv", path="./data/"
    )

In [31]:
config = Tacotron2Config(
        batch_size=64,
        eval_batch_size=16,
        num_loader_workers=0,
        num_eval_loader_workers=0,
        run_eval=False,
        test_delay_epochs=-1,
        epochs=1000,
        print_step=25,
        print_eval=False,
        mixed_precision=True,
        output_path="train_data1",
        datasets=[dataset_config],
         test_sentences=[
                "Мне потребовалось довольно много времени, чтобы развить голос, и теперь я не собираюсь молчать.",
                "Будь голосом, а не эхом.",
                "Мне жаль, Дэйв. Боюсь, я не могу этого сделать.",
                "Этот пирог просто великолепен. Он такой вкусный и сочный.",
                "До ноября года.",
            ]
    )

    # INITIALIZE THE AUDIO PROCESSOR
    # Audio processor is used for feature extraction and audio I/O.
    # It mainly serves to the dataloader and the training loggers.
ap = AudioProcessor.init_from_config(config)

    # LOAD DATA SAMPLES
    # Each sample is a list of ```[text, audio_file_path, speaker_name]```
    # You can define your custom sample loader returning the list of samples.
    # Or define your custom formatter and pass it to the `load_tts_samples`.
    # Check `TTS.tts.datasets.load_tts_samples` for more details.
train_samples, eval_samples = load_tts_samples(
    dataset_config,
    eval_split=True,
    eval_split_size=0.1,
)

    # INITIALIZE THE MODEL
    # Models take a config object and a speaker manager as input
    # Config defines the details of the model like the number of layers, the size of the embedding, etc.
    # Speaker manager is used by multi-speaker models.
model = Tacotron2(config, ap, tokenizer, speaker_manager=None)

In [28]:
audio_config = BaseAudioConfig(
    sample_rate=16000,
    do_trim_silence=False,
    trim_db=60.0,
    signal_norm=False,
    mel_fmin=0.0,
    mel_fmax=8000,
    spec_gain=1.0,
    log_func="np.log",
    ref_level_db=20,
    preemphasis=0.0,
)

config = Tacotron2Config(
    audio=audio_config,
    run_eval=True,
    test_delay_epochs=-1,
    ga_alpha=0.0,              # Это веса функций потерь такотрона
    decoder_loss_alpha=0.25,   # Это веса функций потерь такотрона
    postnet_loss_alpha=0.25,   # Это веса функций потерь такотрона
    postnet_diff_spec_alpha=0, # Это веса функций потерь такотрона
    decoder_diff_spec_alpha=0, # Это веса функций потерь такотрона
    decoder_ssim_alpha=0,      # Это веса функций потерь такотрона
    postnet_ssim_alpha=0,      # Это веса функций потерь такотрона
    optimizer='Adam',          
    grad_clip=1.0,
    r=2,                       # Это количество фреймов, предсказываемых за раз; меньше -- быстрее сходится!
    max_decoder_steps=500,     # Вот это нужно понижать, чтобы инференс был быстрее
    attention_type="original", # Вот это минорная вариация в архитектуре Такотрона
    double_decoder_consistency=False,     # Это тоже
    epochs=1000,
    batch_size=64,
    eval_batch_size=16,
    lr=1e-5,                              # LR поменьше, чтобы поглубэе спустилося
    lr_scheduler=None,                    # По умолчанию там странный шедулер, чтобы не экспериментировать, лучше старый добрый конст
    print_step=25,
    print_eval=True,
    mixed_precision=False,
    datasets=[dataset_config],
    max_audio_len=160000,                # Чтобы не падало по памяти,если слишком длинная вавка попадает в бач.
    test_sentences=[
                "Мне потребовалось довольно много времени, чтобы развить голос, и теперь я не собираюсь молчать.",
                "Будь голосом, а не эхом.",
                "Мне жаль, Дэйв. Боюсь, я не могу этого сделать.",
                "Этот пирог просто великолепен. Он такой вкусный и сочный.",
                "До ноября года.",
            ]
)

ap = AudioProcessor.init_from_config(config)

    # LOAD DATA SAMPLES
    # Each sample is a list of ```[text, audio_file_path, speaker_name]```
    # You can define your custom sample loader returning the list of samples.
    # Or define your custom formatter and pass it to the `load_tts_samples`.
    # Check `TTS.tts.datasets.load_tts_samples` for more details.
train_samples, eval_samples = load_tts_samples(
    dataset_config,
    eval_split=True,
    eval_split_size=0.1,
)

    # INITIALIZE THE MODEL
    # Models take a config object and a speaker manager as input
    # Config defines the details of the model like the number of layers, the size of the embedding, etc.
    # Speaker manager is used by multi-speaker models.
model = Tacotron2(config, ap, tokenizer, speaker_manager=None)

In [33]:
import gc
gc.collect()
torch.cuda.empty_cache()

Загрузим модель из наиболее актуального чекпоинта:

In [178]:
model.load_checkpoint(config, "train_data\\run-December-27-2025_08+25PM-0000000\\best_model.pth")

In [181]:
synthesized = model.synthesize(text="Мы всем отрядом съели все, что было во всем лагере, и все остались довольны.", use_griffin_lim=True)["wav"]
synthesized.shape

(156416,)

In [182]:
Audio(synthesized, rate=16000)

И синтезируем речь по заданным фразам с её помощью:

In [184]:
texts = [
    "Мой слуга, повар и спутник по охоте — полесовщик Ярмола вошел в комнату, согнувшись под вязанкой дров, сбросил ее с грохотом на пол и подышал на замерзшие пальцы.",
    "У, какой ветер, паныч, на дворе, — сказал он, садясь на корточки перед заслонкой.",
    "Нужно хорошо в грубке протопить. Позвольте запалочку, паныч.",
    "Значит, завтра на зайцев не пойдем, а?",
    "Как ты думаешь, Ярмола?",
    "В анамнезе супермена был синтезированный компьютером криптонит.",
    "Мы всем отрядом съели все, что было во всем лагере, и все остались довольны.",
    "Бык тупогуб, тупогубенький бычок, У быка бела губа была тупа.",
    "В день Тлакашипеуалицтли жители Теночтитлана славят Шипе-Тотека, но не забывают также и про Уицилопочтли, Кецалькоатля и Тескатлипоку.",
    "Вот оно что. А он что? Да ты что!"
]

In [185]:
for text in texts:
    synthesized = model.synthesize(text=text, use_griffin_lim=True)["wav"]
    print(text)
    display(Audio(synthesized, rate=16000))

Мой слуга, повар и спутник по охоте — полесовщик Ярмола вошел в комнату, согнувшись под вязанкой дров, сбросил ее с грохотом на пол и подышал на замерзшие пальцы.


У, какой ветер, паныч, на дворе, — сказал он, садясь на корточки перед заслонкой.


Нужно хорошо в грубке протопить. Позвольте запалочку, паныч.


Значит, завтра на зайцев не пойдем, а?


Как ты думаешь, Ярмола?


В анамнезе супермена был синтезированный компьютером криптонит.


Мы всем отрядом съели все, что было во всем лагере, и все остались довольны.


Бык тупогуб, тупогубенький бычок, У быка бела губа была тупа.


В день Тлакашипеуалицтли жители Теночтитлана славят Шипе-Тотека, но не забывают также и про Уицилопочтли, Кецалькоатля и Тескатлипоку.


Вот оно что. А он что? Да ты что!


## Выводы

В результате обучения модели удалось получить относительно разборчивую речь. Слышимые неточности в речи можно обосновать ошибками модели, преобразующей графемы в аллофоны (WRR генерации аллофонов составляет 58,8%, из-за чего модель не всегда правильно расставляет ударения и смягчает звуки, а WRR генерации фонем - 76,6%, из-за чего модель иногда произносит неправильные, лишние звуки или пропускает их).